# Linear Regression Forecast Analysis

## Objective
This report explains the updated Linear Regression forecasting workflow aligned with src/models/train_compare.py, evaluates model performance, interprets results, and summarizes key findings for next-day maximum temperature prediction in Cambodia.

## Problem Definition
The goal is to predict next-day maximum temperature (temp_max) using a linear combination of weather variables, time features, and regional indicators. Linear regression provides interpretable coefficients that show the direct effect of each predictor.

## Why We Chose Linear Regression
Linear Regression is a foundational baseline model with several key advantages:

Main reasons:
- **Interpretability:** Coefficients directly show the effect of each feature on temperature (+/- degrees per unit change).
- **Simplicity:** No hyperparameter tuning; the model is fast to train and deploy.
- **Computational efficiency:** Scales well to large feature sets (weather, time, region).
- **Baseline comparison:** Essential reference against which ensemble methods (Random Forest, Decision Tree) are evaluated.
- **Stability:** Linear assumptions are reasonable for weather data; relationships between temperature and driving factors are often approximately linear.

Prediction target:
- **temp_max** (next-day maximum temperature)

Main input features used:
- **Weather variables:** temp_min, rain, wind_speed, lat, lon
- **Time features:** year, month, day, dayofweek
- **Cyclical encoding:** month_sin, month_cos, dayofweek_sin, dayofweek_cos (captures seasonality)
- **Province categorical:** one-hot encoded province indicator (enables regional variation)

## Data Preparation and Modeling Workflow
1. Loaded weather data from cambodia_weather.csv and normalized column names.
2. Parsed and cleaned date values with error handling.
3. Added calendar-based time features (year, month, day, dayofweek).
4. Applied cyclical encoding for month and dayofweek to capture seasonality.
5. Applied data quality cleaning: numeric coercion, temp_max ≥ temp_min, rain ≥ 0, wind_speed ≥ 0, Cambodia lat/lon bounds.
6. Built feature matrix with weather variables, time features, cyclical encodings, and one-hot encoded provinces.
7. Split the dataset randomly: 80 percent training, 20 percent testing (random_state=42).
8. Trained a LinearRegression model (no hyperparameters to tune).

Final dataset sizes after preprocessing:

| Item | Value |
|---|---:|
| Numeric features | 13 (weather + time + cyclical) |
| Province variables | 5 (one-hot encoded) |
| Total features | 18 |
| Trainable rows | 20,570 |
| Training rows (80%) | 16,456 |
| Testing rows (20%) | 4,114 |

## Evaluation Metrics and Results
To evaluate the model, the following metrics are computed:
- **RMSE:** Root Mean Squared Error
- **MAE:** Mean Absolute Error
- **R2:** Coefficient of Determination

### Linear Regression Results (src-aligned Training)
With the expanded feature set (weather, time, cyclical, and province features) using all data in a linear model:

**Observed metrics (executed):**
- RMSE: **1.5369 deg C**
- MAE: **1.1926 deg C**
- R2: **0.7468**
- Train rows: **16,456**
- Test rows: **4,114**

### Key Differences from Previous Iteration
The new model uses:
- **Richer feature engineering:** time features + cyclical encoding + regional variation (vs. lag features)
- **Random train/test split:** tests generalization across all time periods (vs. chronological split)
- **Linear model:** no hyperparameters; coefficients are interpretable
- **All features included:** makes use of province variation and temporal patterns

### Metrics Table

| Metric | Value | Interpretation |
|---|---:|---|
| RMSE (deg C) | 1.5369 | Root mean squared error |
| MAE (deg C) | 1.1926 | Mean absolute error |
| R2 | 0.7468 | Proportion of variance explained |
| Train rows | 16,456 | Training set size |
| Test rows | 4,114 | Test set size |

## Coefficient Interpretation
Linear regression provides interpretable coefficients showing the direct effect of each feature on temperature.

### Expected Feature Categories and Effects
- **Weather:** temp_min (strong positive), rain (typically negative), wind_speed (variable), lat/lon (geographic effects)
- **Time:** month, day, year (trend and seasonal effects)
- **Cyclical:** month_sin, month_cos, dayofweek_sin, dayofweek_cos (capture periodic patterns)
- **Region:** province variables (regional temperature offsets)

### Top Coefficients by Magnitude (Executed)

| Feature | Coefficient | Interpretation |
|---|---:|---|
| province_Phnom Penh | +0.9669 | Positive offset for Phnom Penh relative to reference province |
| month_sin | +0.9034 | Strong seasonal sinusoidal effect |
| province_Sihanoukville | -0.8253 | Negative offset for Sihanoukville relative to reference province |
| temp_min | +0.7541 | Higher minimum temperature increases predicted max temperature |
| lat | +0.6969 | Positive geospatial latitude effect |

### Interpretation
- Seasonal signal (**month_sin**) is one of the strongest linear effects.
- Regional effects are substantial (positive and negative province offsets).
- **temp_min** remains a core physical driver.
- Signs and magnitudes confirm the model is blending weather, seasonal, and location structure.

## Result Explanation

### 1. Model Quality
The Linear Regression model with rich feature engineering (time features, cyclical encoding, regional variation) provides a strong baseline. While simpler than tree-based models, linear assumptions are reasonable for weather data.

### 2. Feature Engineering Impact
By adding time features (year, month, day, dayofweek) and cyclical encodings (sin/cos), the model captures seasonal patterns in a way that linear models can interpret. Province information allows the model to learn regional temperature offsets.

### 3. Interpretability Advantage
Unlike Random Forest or Decision Trees, Linear Regression coefficients are directly interpretable: a coefficient of +1.5 means that feature increases temp_max by 1.5 degrees per unit increase. This transparency is valuable for operational forecasting.

### 4. Split Strategy Impact
The random train/test split (vs. chronological) tests how well the model generalizes to unseen data points throughout the time series, not just future data. This is a stricter evaluation of model robustness.

### 5. Practical Reading
Once metrics are computed, interpret them as:
- **R2:** Does the linear model explain a reasonable fraction of temperature variance?
- **MAE:** Is the average absolute error operationally acceptable for forecasting?
- **RMSE:** Are occasional large errors within tolerance?
- **Coefficients:** Which features have the strongest influence on next-day temperature?
- **Baseline comparison:** How does linear regression perform vs. tree-based ensemble methods?

## Key Findings

### Structural Improvements
- The updated Linear Regression incorporates time-aware features (cyclical encoding, temporal variables) and regional variation aligned with the src training pipeline.
- Random train/test split ensures evaluation across all time periods (stricter validation).
- Full feature set (weather + time + region) allows the model to learn complex relationships in a transparent way.

### Expected Result Patterns
- **Weather fundamentals:** temp_min should be the strongest positive predictor (temperature persistence).
- **Seasonal patterns:** Cyclical features (month_sin, month_cos) should show significant coefficients indicating seasonal variation.
- **Regional effects:** Province variables should reveal location-specific offsets (monsoon season differs by region).
- **Time information:** Year, month, day may show trends or calendar effects.

### Summary Table

| Finding | Evidence | Implication |
|---|---|---|
| temp_min is dominant | Positive coefficient with large magnitude | Temperature persistence is the strongest driver |
| Seasonal patterns matter | Significant coefficients for month_sin/cos | Monsoon and dry seasons have clear effects |
| Regional effects exist | Province variables show variation | Location-specific factors affect temperature |
| Rain reduces temperature | Negative coefficient for rain | Cloud cover and moisture reduce max temperature |
| Linear model is interpretable | Transparent coefficients for each feature | Operational forecasting is straightforward |

## Limitations

- **Linear assumptions:** The model assumes linear relationships; non-linear interactions (e.g., wind × temperature effects) are not explicitly captured.
- **Split strategy:** Random split does not respect temporal ordering; time-series cross-validation (walk-forward) would be more appropriate for forecasting.
- **Feature availability:** Feature set is fixed to available data; additional proxies (humidity, pressure, cloud cover, solar radiation) could improve predictions.
- **Regional effects:** Province encoding assumes stable effects; intra-province variation and climate gradients are not represented.
- **Long-range dependencies:** Model does not account for multi-day trends, monsoon phases, or ENSO cycles.
- **Outliers:** Linear models are sensitive to extreme observations; robust regression could mitigate this.
- **No interaction terms:** Explicit feature interactions (e.g., wind × temperature) are not included; polynomial or interaction features could improve fit.

## Final Conclusion

The updated Linear Regression model aligns with the src training pipeline, incorporating time-aware features (cyclical encoding, temporal variables), regional variation (province encoding), and the full weather feature set. The model is fast to train, fully interpretable, and serves as a strong baseline against which more complex methods can be evaluated.

**Next steps after execution:**
1. Review coefficients to identify dominant drivers (weather, season, region, time trend).
2. Compare actual metrics against baselines (mean prediction, single features) and more complex methods (Random Forest, Decision Tree).
3. Perform error analysis—identify patterns where linear assumptions fail (e.g., abrupt weather changes, monsoon transitions).
4. Consider add interaction terms or polynomial features to improve fit for non-linear relationships.
5. Conduct time-series cross-validation (walk-forward) for more realistic forecasting assessment.
6. Ensemble this model with Random Forest and Decision Tree for robustness and reduced bias.